# Aprendizado de Máquina — Lista prática 02

## Regressão Linear e Regularização

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O `superconductivity.csv` tem 21 263 supercondutores e 81 covariáveis. Com todas
as observações, $n$ é 260 vezes maior que $d$ e o mínimos quadrados vai muito bem
— não há o que regularizar. Esta lista faz o contrário: fica com **100
observações de treino**, para pôr você exatamente no regime em que a Aula 02
mora, com $n$ pouco maior que $d$.

> **quando $n$ e $d$ são comparáveis, o mínimos quadrados não fica só um pouco
> pior — ele quebra.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — os dados e a divisão

A célula de carga é a mesma da aula prática. Complete a separação da resposta e
a divisão treino/teste.

A resposta é `critical_temp`, a temperatura crítica em kelvin.

In [ ]:
_nome = "superconductivity.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

df = pd.read_csv(_fonte)
X_todos = df.drop(columns=...)                    # (a)
y_todos = df[...].values                          # (b)

print("dimensoes:", X_todos.shape)

Agora a divisão. Peça **100** observações de treino e 5 000 de teste — o resto do
banco fica de fora de propósito. A semente é 2026.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X_todos.values, y_todos,
    train_size=...,                                           # (a)
    test_size=5000,
    random_state=...,                                        # (b)
)

print(f"treino: {X_tr.shape[0]} observacoes, {X_tr.shape[1]} covariaveis")
print(f"teste:  {X_te.shape[0]} observacoes")

---
## Exercício 2 — o mínimos quadrados quebra

Ajuste o MQO e meça o $R^2$ nos **dois** conjuntos. O `Pipeline` padroniza antes
de ajustar; para o MQO isso não muda nada (Exercício 2(c) da lista teórica), mas
deixa o código pronto para o Ridge e o Lasso.

In [ ]:
def tubo(modelo):
    return Pipeline([("escala", StandardScaler()), ("mod", modelo)])


mqo = tubo(...).fit(X_tr, y_tr)            # (a)

print(f"MQO   R2 treino {mqo.score(X_tr, y_tr):.4f}")
print(f"MQO   R2 teste  {...:.4f}")         # (b)

**Responda** na célula abaixo, como comentário: o que significa um $R^2$
negativo, e por que ele apareceu aqui?

---
## Exercício 3 — Ridge e Lasso no mesmo treino

Sem trocar uma única observação, ajuste os dois métodos penalizados. Use
$\alpha=10$ na Ridge e $\alpha=1$ no Lasso.

Conte também **quantos coeficientes cada um deixou diferentes de zero** — é a
diferença que a lista teórica previu na aritmética.

In [ ]:
for nome, modelo in [("Ridge", ...),                   # (a)
                     ("Lasso", ...)]:   # (b)
    ajuste = tubo(modelo).fit(X_tr, y_tr)
    coef = ajuste.named_steps["mod"].coef_
    nao_nulos = ...                        # (c)
    print(f"{nome}  R2 treino {ajuste.score(X_tr, y_tr):.4f}   "
          f"R2 teste {ajuste.score(X_te, y_te):.4f}   "
          f"coefs != 0: {nao_nulos}/81")

> **Sua vez.** Repita o Exercício 2 e este, trocando `train_size=100` por
> `train_size=2000`. O MQO continua quebrado? E a vantagem do Lasso sobre ele,
> continua existindo?

---
## Exercício 4 — o caminho do Lasso e a escolha de $\alpha$

O $\alpha=1$ do exercício anterior foi um chute. Percorra uma grade de $\alpha$ e
guarde, para cada um, quantos coeficientes sobrevivem e qual o $R^2$ de teste.

In [ ]:
alphas = ...                             # (a) de 0,01 a ~31,6
n_coefs, r2_teste = [], []

for a in alphas:
    ajuste = tubo(skl.Lasso(alpha=a, max_iter=20000)).fit(X_tr, y_tr)
    coef = ajuste.named_steps["mod"].coef_
    n_coefs.append(int(np.sum(np.abs(coef) > 1e-10)))
    r2_teste.append(...)                 # (b)

for a, k, r2 in zip(alphas, n_coefs, r2_teste):
    print(f"alpha {a:8.4f}:  {k:2d} coefs,  R2 teste {r2:7.4f}")

Desenhe o caminho. Como os $\alpha$ variam em ordens de grandeza, o eixo
horizontal precisa ser logarítmico.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(9, 3.2))

ax1.plot(alphas, n_coefs, "o-", ms=4)
ax1.set_xscale(...)                                         # (a)
ax1.set_xlabel(r"$\alpha$")
ax1.set_ylabel("coeficientes diferentes de zero")

ax2.plot(alphas, r2_teste, "o-", ms=4)
ax2.set_xscale("log")
ax2.set_ylim(..., 0.75)                                      # (b) corta o -1,97
ax2.set_xlabel(r"$\alpha$")
ax2.set_ylabel("$R^2$ de teste")

fig.tight_layout()

Agora escolha o $\alpha$ **sem olhar o teste** — que é a única forma honesta.
Use validação cruzada de 5 dobras sobre o treino.

In [ ]:
busca = skm.GridSearchCV(
    tubo(skl.Lasso(max_iter=20000)),
    {...: alphas},                                   # (a) o nome do passo, dois underscores
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring=...,                         # (b)
).fit(X_tr, y_tr)

escolhido = busca.best_params_["mod__alpha"]
melhor = busca.best_estimator_
coef = melhor.named_steps["mod"].coef_

print(f"CV escolheu alpha = {escolhido:.4f}")
print(f"  coeficientes != 0: {int(np.sum(np.abs(coef) > 1e-10))}/81")
print(f"  R2 de teste:       {melhor.score(X_te, y_te):.4f}")

Por fim, veja **quais** covariáveis o Lasso manteve, ordenadas por magnitude.

In [ ]:
nomes = np.array(X_todos.columns)
ordem = ...                         # (a) as 5 de maior módulo

for j in ordem:
    print(f"{nomes[j]:42s} {coef[j]:8.3f}")

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 2 | com $n=100$ e $d=81$, o MQO dá $R^2$ de treino 0,9582 e de teste **−19,09** |
| 3 | Ridge e Lasso levam o mesmo treino a $R^2\approx 0{,}64$ — e o Lasso usa 13 covariáveis |
| 4 | o caminho vai de 66 coeficientes ($\alpha=0{,}01$) a nenhum ($\alpha=31{,}6$) |
| 4 | a CV escolhe $\alpha=0{,}3162$ e perde 0,02 de $R^2$ para o melhor da grade — sem olhar o teste |

**A seguir.** A Aula 03 formaliza o que o último exercício fez à mão: como estimar
risco e escolher hiperparâmetro sem gastar um conjunto de teste, e como reportar
desempenho depois de ter escolhido.